 # 02 - Feature Pipeline

Builds the model-ready feature matrix from the raw store: returns, rolling volatility, momentum, and the point-in-time macro block, followed by the outlier screen and the date-based train/validation/test split. Every parameter comes from `config/config.yaml` so the split dates are identical across all experiments. Outputs are written once and read by notebooks 03-05.

Four leakage guards are enforced here:

1. **Point-in-time macro data.** Most FRED series (CPIAUCSL above all) carry a publication lag. Using the same-day value would hand the model information no trader had on the day; the whole macro block is shifted by `features.macro_lag_bdays` business days before joining.
2. **Next-day labels.** The supervised target is the *next-day* return, matching the T+1 trade convention - row *t* never peeks at its own close-to-close move.
3. **Train-only winsorization bounds.** Clip quantiles are estimated on the training block only, so test-period tails cannot leak into any feature used for fitting.
4. **Joint row mask.** Rows are dropped only after features and labels are joined, so the two always describe exactly the same sample.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import numpy as np
import pandas as pd

from src.cleaning import set_seed, winsorize_with_bounds
from src.config import load_config, project_root

CFG = load_config()
ROOT = project_root()

# Reproducibility pre-commitment: one shared code path pins the python and
# numpy generators from config/config.yaml before any data is touched.
set_seed(int(CFG["seeds"]["numpy"]))

prices = pd.read_csv(ROOT / "data/raw/prices.csv", index_col=0, parse_dates=True)
macro = pd.read_csv(ROOT / "data/raw/macro.csv", index_col=0, parse_dates=True)

# Calendar alignment across tickers (bounded ffill per the cleaning policy).
prices = prices.sort_index().ffill(limit=CFG["cleaning"]["ffill_limit"]).dropna(how="all")
print("Aligned price matrix:", prices.shape)

 ## 1. Core features

Daily simple returns feed the LSTM directly; annualized rolling volatility and multi-horizon momentum capture the persistence the networks are expected to exploit (Problem Statement, Section 3.1).

All features on row *t* use data up to and including the close of day *t* only. The supervised target is the **next-day** return (`targets`), consistent with the T+1 allocation convention. Rows with incomplete history or a missing label are dropped *after* the join - the count is printed so the cleaning log can cite it.

In [ ]:
rets = prices.pct_change()   # keep NaNs; the row mask is applied after the join
vol = rets.rolling(CFG["features"]["vol_window"]).std() * np.sqrt(252)
mom = rets.rolling(CFG["features"]["mom_window"]).sum()

# Macro block: point-in-time shift for publication lag, then align to the
# trading calendar. Default lag of 14 business days covers CPIAUCSL (the
# slowest series in the set); fast series (VIXCLS, DFF) are checked again
# against their real release calendars at M3 and given per-series lags.
MACRO_LAG_BDAYS = int(CFG["features"].get("macro_lag_bdays", 14))
macro = macro.sort_index().ffill(limit=CFG["cleaning"]["ffill_limit"])
macro = macro.shift(MACRO_LAG_BDAYS).reindex(rets.index)

# One wide frame -> MultiIndex columns (block, ticker). Rows are dropped
# only AFTER the join, so features and labels share exactly the same
# sample: no silent NaNs reach the models in notebook 03.
panel = pd.concat({"ret": rets, "vol": vol, "mom": mom, "macro": macro}, axis=1)

# Target: next-day return, aligned so row t carries E[r_{t+1} | info_t].
targets_all = panel.xs("ret", axis=1, level=0).shift(-1)
usable = panel.notna().all(axis=1) & targets_all.notna().all(axis=1)
print(f"Rows dropped for incomplete history/labels: {(~usable).sum()} of {len(panel)}")
panel, targets = panel[usable], targets_all[usable]
print("Feature blocks:", panel.columns.get_level_values(0).unique().tolist())
print("Sample:", panel.index.min().date(), "->", panel.index.max().date(), len(panel))

 ## 2. Outlier screen

Winsorization at the configured quantiles, with the rule fixed before the test block is touched (pre-commitment against data snooping).

Two refinements over a naive screen: the clip bounds are **estimated on the training block only** (applying full-sample quantiles would smuggle test-period extremes into the fitted features), and they are computed **per column** via `src.cleaning.winsorize_with_bounds` - a single cross-sectional bound would let one volatile name drag the tails of every quiet sector ETF toward its own distribution.

In [ ]:
q_lo, q_hi = CFG["cleaning"]["winsor"]
# Bounds come from the TRAINING BLOCK ONLY (see markdown above); the clip
# itself is the shared per-column policy from src.cleaning.
tr = panel.index <= pd.Timestamp(CFG["splits"]["train_end"])
screened, lo, hi = winsorize_with_bounds(panel, q_lo, q_hi, fit=tr)
targets, t_lo, t_hi = winsorize_with_bounds(targets, q_lo, q_hi, fit=tr)
n_clipped = int((panel != screened).sum().sum())
print("Values clipped (train-bounded):", n_clipped)
# Record the fitted bounds themselves so the report can cite exact numbers.
bounds_rows = pd.DataFrame(
    {"col": lo.index, "lo": lo.values, "hi": hi.values}
).assign(block="features")


## 3. Date-based splits

Train/validation/test boundaries come from the config, not from data. The test block remains untouched until the final walk-forward run.

In [ ]:
splits = CFG["splits"]
train = screened.loc[splits["train_start"]:splits["train_end"]]
valid = screened.loc[splits["valid_start"]:splits["valid_end"]]
test = screened.loc[splits["test_start"]:]
for name, frame in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name}: {frame.index.min().date()} -> {frame.index.max().date()}  rows={len(frame)}")
assert not test.empty, "Test block empty - check split dates in config."
# Guardrail: the blocks must be strictly ordered and non-overlapping.
assert train.index.max() < valid.index.min()
assert valid.index.max() < test.index.min()

## 4. Persist

Parquet is used for the feature matrix to preserve dtypes. The cleaning log below is the artifact the report cites.

In [ ]:
os.makedirs(ROOT / "data/processed", exist_ok=True)
screened.to_parquet(ROOT / "data/processed/features.parquet")
targets.to_parquet(ROOT / "data/processed/targets.parquet")
screened.xs("macro", axis=1, level=0).to_csv(ROOT / "data/processed/macro_aligned.csv")
log = pd.DataFrame({
    "step": ["ffill_limit", "winsor_low", "winsor_high", "macro_lag_bdays",
             "winsor_bounds_source", "values_clipped", "rows", "tickers"],
    "value": [CFG["cleaning"]["ffill_limit"], q_lo, q_hi, MACRO_LAG_BDAYS,
              "train-only", n_clipped, len(screened), prices.shape[1]],
})
log.to_csv(ROOT / "data/processed/cleaning_log.csv", index=False)
pd.concat([
    bounds_rows,
    pd.DataFrame({"col": t_lo.index, "lo": t_lo.values, "hi": t_hi.values}).assign(block="targets"),
]).to_csv(ROOT / "data/processed/winsor_bounds.csv", index=False)
print(log)